In [1]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

print("Validation notebook started successfully.")

Validation notebook started successfully.


In [2]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor

# Load sales data
sales = pd.read_csv("../data/raw/sales_daily.csv")

# Convert date
sales["Date"] = pd.to_datetime(
    sales["Date"],
    errors="coerce"
)

# Create weekly demand
sales["Week"] = (
    sales["Date"]
    .dt.to_period("W")
    .dt.end_time
    .dt.normalize()
)

weekly_sku_demand = (
    sales
    .groupby(["Week", "SKU"], as_index=False)["Units_Sold"]
    .sum()
    .sort_values(["SKU", "Week"])
    .reset_index(drop=True)
)

# Load calendar data
calendar = pd.read_csv("../data/raw/calendar.csv")

calendar["date"] = pd.to_datetime(
    calendar["date"],
    errors="coerce"
)

# Clean promotion values
calendar["promotion_event"] = (
    calendar["promotion_event"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        "nan": "0",
        "none": "0",
        "": "0",
        "yes": "1",
        "true": "1",
        "no": "0",
        "false": "0"
    })
)

calendar["promotion_event"] = pd.to_numeric(
    calendar["promotion_event"],
    errors="coerce"
).fillna(0)

calendar["is_holiday"] = pd.to_numeric(
    calendar["is_holiday"],
    errors="coerce"
).fillna(0)

# Create weekly calendar information
calendar["Week"] = (
    calendar["date"]
    .dt.to_period("W")
    .dt.end_time
    .dt.normalize()
)

calendar_weekly = (
    calendar
    .groupby("Week", as_index=False)
    .agg(
        Promotion_Event=("promotion_event", "max"),
        Holiday_Count=("is_holiday", "sum")
    )
)

# Merge demand and calendar information
features = weekly_sku_demand.merge(
    calendar_weekly,
    on="Week",
    how="left"
)

features["Promotion_Event"] = (
    features["Promotion_Event"].fillna(0)
)

features["Holiday_Count"] = (
    features["Holiday_Count"].fillna(0)
)

# Create lag features
features = features.sort_values(
    ["SKU", "Week"]
).reset_index(drop=True)

features["Lag_1"] = (
    features.groupby("SKU")["Units_Sold"].shift(1)
)

features["Lag_2"] = (
    features.groupby("SKU")["Units_Sold"].shift(2)
)

features["Lag_4"] = (
    features.groupby("SKU")["Units_Sold"].shift(4)
)

features["Lag_52"] = (
    features.groupby("SKU")["Units_Sold"].shift(52)
)

# Create rolling features
features["Rolling_4_Week"] = (
    features.groupby("SKU")["Units_Sold"]
    .transform(lambda x: x.shift(1).rolling(4).mean())
)

features["Rolling_8_Week"] = (
    features.groupby("SKU")["Units_Sold"]
    .transform(lambda x: x.shift(1).rolling(8).mean())
)

# Remove rows with unavailable lag or rolling values
features_clean = features.dropna(
    subset=[
        "Lag_1",
        "Lag_2",
        "Lag_4",
        "Lag_52",
        "Rolling_4_Week",
        "Rolling_8_Week"
    ]
).reset_index(drop=True)

print("Feature dataset recreated successfully.")
print("Rows:", len(features_clean))
print("Columns:", len(features_clean.columns))

Feature dataset recreated successfully.
Rows: 2650
Columns: 11


In [3]:
from sklearn.ensemble import RandomForestRegressor

feature_columns = [
    "Lag_1",
    "Lag_2",
    "Lag_4",
    "Lag_52",
    "Rolling_4_Week",
    "Rolling_8_Week",
    "Promotion_Event",
    "Holiday_Count"
]

# Use the final 36 available weeks as validation period
all_weeks = sorted(features_clean["Week"].unique())

validation_weeks = all_weeks[-36:]

# Evaluate at three rolling forecast origins
origins = [
    validation_weeks[0],
    validation_weeks[12],
    validation_weeks[24]
]

validation_results = []

for origin_week in origins:
    train_data = features_clean[
        features_clean["Week"] < origin_week
    ].copy()

    test_data = features_clean[
        features_clean["Week"] >= origin_week
    ].copy()

    # Limit each validation window to 12 weeks
    test_weeks = sorted(test_data["Week"].unique())[:12]

    test_data = test_data[
        test_data["Week"].isin(test_weeks)
    ].copy()

    if train_data.empty or test_data.empty:
        continue

    X_train = train_data[feature_columns]
    y_train = train_data["Units_Sold"]

    X_test = test_data[feature_columns]
    y_test = test_data["Units_Sold"]

    model = RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    predictions = np.maximum(predictions, 0)

    absolute_error = np.abs(
        y_test.to_numpy() - predictions
    )

    model_wape = (
        absolute_error.sum() / y_test.sum()
    )

    baseline_predictions = test_data["Lag_52"]

    baseline_error = np.abs(
        y_test.to_numpy() -
        baseline_predictions.to_numpy()
    )

    baseline_wape = (
        baseline_error.sum() / y_test.sum()
    )

    validation_results.append({
        "Origin_Week": origin_week,
        "Random_Forest_WAPE": model_wape * 100,
        "Seasonal_Naive_WAPE": baseline_wape * 100
    })

validation_results_df = pd.DataFrame(validation_results)

display(validation_results_df)

,Origin_Week,Random_Forest_WAPE,Seasonal_Naive_WAPE
0,2025-05-04,9.171933,9.915286
1,2025-07-27,10.455114,11.766521
2,2025-10-19,15.469555,17.307613


In [4]:
if validation_results_df.empty:
    print("No validation results were generated.")

else:
    average_random_forest_wape = (
        validation_results_df["Random_Forest_WAPE"].mean()
    )

    average_baseline_wape = (
        validation_results_df["Seasonal_Naive_WAPE"].mean()
    )

    print(
        f"Average Random Forest WAPE: "
        f"{average_random_forest_wape:.2f}%"
    )

    print(
        f"Average Seasonal-Naive WAPE: "
        f"{average_baseline_wape:.2f}%"
    )

    if average_random_forest_wape < average_baseline_wape:
        print(
            "Result: Random Forest performs better "
            "than the Seasonal-Naive baseline."
        )
    else:
        print(
            "Result: Seasonal-Naive performs better "
            "than Random Forest."
        )

Average Random Forest WAPE: 11.70%
Average Seasonal-Naive WAPE: 13.00%
Result: Random Forest performs better than the Seasonal-Naive baseline.


In [5]:
# Save detailed validation results
validation_results_df.to_csv(
    "../reports/rolling_origin_validation.csv",
    index=False
)

# Save average performance summary
average_results = pd.DataFrame({
    "Model": [
        "Seasonal-Naive",
        "Random Forest"
    ],
    "Average_WAPE_Percent": [
        average_baseline_wape,
        average_random_forest_wape
    ]
})

average_results.to_csv(
    "../reports/average_model_performance.csv",
    index=False
)

print("Validation reports saved successfully.")
print("Files created:")
print("- rolling_origin_validation.csv")
print("- average_model_performance.csv")

Validation reports saved successfully.
Files created:
- rolling_origin_validation.csv
- average_model_performance.csv
